# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [13]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [14]:
import pandas as pd
from pathlib import Path

In [51]:
DATASET_NAME = "merfish"
N_DEG = 50
CSV_PATH = f"../results/loo_summary_{DATASET_NAME}_DEG_{N_DEG}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,50,0.574783,0.676753,0.00,0.0,0.00,0.08,0.019022,42.347551,43.229744,1522.121790,27.050253,613.618408,398.293091
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,50,0.656297,0.672887,0.20,0.9,0.18,0.60,0.293658,48.110075,52.501921,1118.618372,29.387819,1490.194336,180.199417
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,50,0.301952,0.354013,0.18,1.0,0.18,0.82,0.069700,38.657775,37.840701,1407.842004,17.100988,1868.877075,152.413559
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,50,0.774857,0.715261,0.26,1.0,0.26,0.84,0.000000,64.602134,66.657219,2111.273730,36.885138,2971.760254,154.598236
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,50,0.742315,0.662692,0.06,1.0,0.06,0.86,0.003236,43.568983,43.794907,1866.005750,21.782019,1668.539429,96.198738


In [52]:
# Rename holdout_celltype by replacing the last '_' in with '-'
if DATASET_NAME == "merfish":
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)
else:
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("T_cell", "T-cell", regex=False)

In [53]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc,perturbation
0,C57BL6J-2.036,baseline,GABAergic neuron,50,0.574783,0.676753,0.00,0.0,0.00,0.08,0.019022,42.347551,43.229744,1522.121790,27.050253,613.618408,398.293091,Fiber-tracts
1,C57BL6J-2.036,baseline,GABAergic neuron,50,0.656297,0.672887,0.20,0.9,0.18,0.60,0.293658,48.110075,52.501921,1118.618372,29.387819,1490.194336,180.199417,Isocortex
2,C57BL6J-2.036,baseline,astrocyte,50,0.301952,0.354013,0.18,1.0,0.18,0.82,0.069700,38.657775,37.840701,1407.842004,17.100988,1868.877075,152.413559,Fiber-tracts
3,C57BL6J-2.036,baseline,astrocyte,50,0.774857,0.715261,0.26,1.0,0.26,0.84,0.000000,64.602134,66.657219,2111.273730,36.885138,2971.760254,154.598236,Isocortex
4,C57BL6J-2.036,baseline,endothelial cell,50,0.742315,0.662692,0.06,1.0,0.06,0.86,0.003236,43.568983,43.794907,1866.005750,21.782019,1668.539429,96.198738,Fiber-tracts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291,C57BL6J-2.041,spatialprop-pert,astrocyte,50,0.731669,0.758148,0.06,1.0,0.06,0.92,0.072450,38.204006,47.718252,401.915613,19.043097,2542.106700,33.687320,Fiber-tracts
292,C57BL6J-2.041,spatialprop-pert,oligodendrocyte,50,0.425018,0.640706,0.10,1.0,0.10,0.96,0.156538,54.967022,65.053135,492.834454,19.598421,1310.807400,127.696980,Isocortex
293,C57BL6J-2.041,spatialprop-pert,oligodendrocyte,50,0.285378,0.658684,0.04,1.0,0.04,0.84,0.050602,55.231331,65.077148,684.917322,27.405459,6247.942400,24.937560,Fiber-tracts
294,C57BL6J-2.041,spatialprop-pert,endothelial cell,50,0.794862,0.866290,0.28,1.0,0.28,0.98,0.012422,52.677394,60.332082,466.868317,20.071766,2591.789800,66.480200,Isocortex


In [54]:
# Apply log10 on rmse
import numpy as np
df['rmse'] = np.log10(df['rmse'])

In [55]:
# Apply square root to mse_lfc
df['rmse_lfc'] = np.sqrt(df['mse_lfc'])

In [56]:
#df = df[df["model_name"] != "cellina-gat-pert"]

In [57]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    #"spearman": +1,
    "pearson": +1,
    "direction_match_k": +1,
    #"edistance_global": -1,
    "edistance_pca_log": -1,
    #"rmse": -1,
    "rmse_lfc": -1,
    #"avg_rank": -1,
}

PRETTY_METRIC = {
    #"spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "direction_match_k": r"Signed Precision $\uparrow$",
    #"edistance_global": r"E-dist (global) $\downarrow$",
    "edistance_pca_log": r"E-distance $\downarrow$",
    #"rmse": r"RMSE\textsubscript{counts} $\downarrow$",
    "rmse_lfc": r"RMSE\textsubscript{LFC} $\downarrow$",
    #"avg_rank": r"Avg. Rank $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    "mintflow": "MintFlow",
    "cellina-ablated": "Cellina (ablated)",
    "cellina-graph": "Cellina-GAT",
    "cellina": "Cellina",
    "cellina-pert": r"Cellina$_{node-pert}$",
    "cellina-gat-pert": r"Cellina-GAT$_{node-pert}$",
    "spatialprop-pert": r"SpatialProp$_{node-pert}$"
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    "MintFlow",
    "Cellina (ablated)",
    "Cellina",
    "Cellina-GAT",
    r"Cellina$_{node-pert}$",
    r"Cellina-GAT$_{node-pert}$",
    r"SpatialProp$_{node-pert}$",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'CPA', 'scGen', 'Cellina', 'Cellina (ablated)',
       'Cellina-GAT', 'MintFlow', 'Cellina$_{node-pert}$',
       'Cellina-GAT$_{node-pert}$', 'SpatialProp$_{node-pert}$'],
      dtype=object)

In [58]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)
agg

pearson           direction_match_k  \
                                            mean       std              mean   
perturbation model_name                                                        
Fiber-tracts mean shift                 0.360952  0.238833          0.068000   
             CPA                        0.798208  0.149858          0.309333   
             scGen                      0.715875  0.201174          0.149333   
             MintFlow                   0.779929  0.179957          0.205333   
             Cellina (ablated)          0.785785  0.143455          0.265333   
             Cellina                    0.808807  0.152731          0.390667   
             Cellina-GAT                0.804517  0.157653          0.380000   
             Cellina$_{node-pert}$      0.806143  0.152371          0.396000   
             Cellina-GAT$_{node-pert}$  0.807807  0.156162          0.402857   
             SpatialProp$_{node-pert}$  0.646903  0.180874          0.065333   
Isocortex    mean shift                 0.575376  0.191781          0.250667   
             CPA                        0.837973  0.172484          0.534667   
             scGen                      0.820247  0.142527          0.268000   
             MintFlow                   0.836694  0.164855          0.366667   
             Cellina (ablated)          0.834210  0.159035          0.486667   
             Cellina                    0.853065  0.163093          0.564000   
             Cellina-GAT                0.886631  0.142107          0.635714   
             Cellina$_{node-pert}$      0.846902  0.165859          0.526667   
             Cellina-GAT$_{node-pert}$  0.883319  0.139757          0.601429   
             SpatialProp$_{node-pert}$  0.736170  0.095215          0.076000   

                                                 edistance_pca_log            \
                                             std              mean       std   
perturbation model_name                                                        
Fiber-tracts mean shift                 0.069200         24.896145  5.702891   
             CPA                        0.125554          7.249715  2.701666   
             scGen                      0.090354          5.276147  4.384697   
             MintFlow                   0.162035         19.986863  1.493028   
             Cellina (ablated)          0.126596          8.231004  3.532384   
             Cellina                    0.172356          7.965364  1.441834   
             Cellina-GAT                0.176635          8.224835  1.710762   
             Cellina$_{node-pert}$      0.180507          9.272612  1.999629   
             Cellina-GAT$_{node-pert}$  0.184450          8.080159  1.659169   
             SpatialProp$_{node-pert}$  0.069474         22.352469  3.985128   
Isocortex    mean shift                 0.081369         34.252907  3.978706   
             CPA                        0.136584          6.798150  1.955997   
             scGen                      0.161917          6.209470  5.227868   
             MintFlow                   0.161186         19.172766  1.963619   
             Cellina (ablated)          0.148452          8.832297  5.695179   
             Cellina                    0.138605          7.801514  1.262697   
             Cellina-GAT                0.096454          8.758777  1.415902   
             Cellina$_{node-pert}$      0.140340          8.976245  2.129334   
             Cellina-GAT$_{node-pert}$  0.112991          8.582068  1.144796   
             SpatialProp$_{node-pert}$  0.065987         22.450624  2.325465   

                                         rmse_lfc            
                                             mean       std  
perturbation model_name                                      
Fiber-tracts mean shift                 12.960565  3.896250  
             CPA                         7.645226  5.807097  
             scGen                       7.510745

In [59]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.2f} $\\pm$ {std:.2f}" if not pd.isna(std) else f"{mean:.2f}"
    return f"\\textbf{{{s}}}" if bold else s

In [60]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Pearson $\uparrow$  \
perturbation model_name                                            
Fiber-tracts mean shift                          0.36 $\pm$ 0.24   
             CPA                                 0.80 $\pm$ 0.15   
             scGen                               0.72 $\pm$ 0.20   
             MintFlow                            0.78 $\pm$ 0.18   
             Cellina (ablated)                   0.79 $\pm$ 0.14   
             Cellina                    \textbf{0.81 $\pm$ 0.15}   
             Cellina-GAT                         0.80 $\pm$ 0.16   
             Cellina$_{node-pert}$               0.81 $\pm$ 0.15   
             Cellina-GAT$_{node-pert}$           0.81 $\pm$ 0.16   
             SpatialProp$_{node-pert}$           0.65 $\pm$ 0.18   
Isocortex    mean shift                          0.58 $\pm$ 0.19   
             CPA                                 0.84 $\pm$ 0.17   
             scGen                               0.82 $\pm$ 0.14   
             MintFlow                            0.84 $\pm$ 0.16   
             Cellina (ablated)                   0.83 $\pm$ 0.16   
             Cellina                             0.85 $\pm$ 0.16   
             Cellina-GAT                \textbf{0.89 $\pm$ 0.14}   
             Cellina$_{node-pert}$               0.85 $\pm$ 0.17   
             Cellina-GAT$_{node-pert}$           0.88 $\pm$ 0.14   
             SpatialProp$_{node-pert}$           0.74 $\pm$ 0.10   

                                       Signed Precision $\uparrow$  \
perturbation model_name                                              
Fiber-tracts mean shift                            0.07 $\pm$ 0.07   
             CPA                                   0.31 $\pm$ 0.13   
             scGen                                 0.15 $\pm$ 0.09   
             MintFlow                              0.21 $\pm$ 0.16   
             Cellina (ablated)                     0.27 $\pm$ 0.13   
             Cellina                               0.39 $\pm$ 0.17   
             Cellina-GAT                           0.38 $\pm$ 0.18   
             Cellina$_{node-pert}$                 0.40 $\pm$ 0.18   
             Cellina-GAT$_{node-pert}$    \textbf{0.40 $\pm$ 0.18}   
             SpatialProp$_{node-pert}$             0.07 $\pm$ 0.07   
Isocortex    mean shift                            0.25 $\pm$ 0.08   
             CPA                                   0.53 $\pm$ 0.14   
             scGen                                 0.27 $\pm$ 0.16   
             MintFlow                              0.37 $\pm$ 0.16   
             Cellina (ablated)                     0.49 $\pm$ 0.15   
             Cellina                               0.56 $\pm$ 0.14   
             Cellina-GAT                  \textbf{0.64 $\pm$ 0.10}   
             Cellina$_{node-pert}$                 0.53 $\pm$ 0.14   
             Cellina-GAT$_{node-pert}$             0.60 $\pm$ 0.11   
             SpatialProp$_{node-pert}$             0.08 $\pm$ 0.07   

                                         E-distance $\downarrow$  \
perturbation model_name                                            
Fiber-tracts mean shift                         24.90 $\pm$ 5.70   
             CPA                                 7.25 $\pm$ 2.70   
             scGen                      \textbf{5.28 $\pm$ 4.38}   
             MintFlow                           19.99 $\pm$ 1.49   
             Cellina (ablated)                   8.23 $\pm$ 3.53   
             Cellina                             7.97 $\pm$ 1.44   
             Cellina-GAT                         8.22 $\pm$ 1.71   
             Cellina$_{node-pert}$               9.27 $\pm$ 2.00   
             Cellina-GAT$_{node-pert}$           8.08 $\pm$ 1.66   
             SpatialProp$_{node-pert}$          22.35 $\pm$ 3.99   
Isocortex    mean shift                         34.25 $\pm$ 3.98   
             CPA                                 6.80 $\pm$ 1.96   
             scGen                      \textbf{6.21 $\p

In [61]:
def insert_midrule_perts(latex, table):
    groups = table.index.get_level_values(0)
    boundaries = [i for i in range(1, len(groups)) if groups[i] != groups[i - 1]]

    lines = latex.splitlines()
    new_lines = []

    row_idx = 0

    for line in lines:
        new_lines.append(line)

        # detect data rows (skip header)
        if "&" in line and "\\\\" in line:
            if row_idx in boundaries:
                new_lines.append(r"\midrule")
            row_idx += 1

    latex = "\n".join(new_lines)
    return latex

In [62]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
    latex = insert_midrule_perts(latex, table)
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")

print(latex)

(OUT_DIR / f"loo_summary_{DATASET_NAME}_DEG_{N_DEG}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 50 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 3 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{llcccc}
\toprule
Holdout \\ perturbation & Method & Pearson $\uparrow$ & Signed Precision $\uparrow$ & E-distance $\downarrow$ & RMSE\textsubscript{LFC} $\downarrow$ \\
\toprule

\multirow[c]{10}{*}{Fiber-tracts} & mean shift & 0.36 $\pm$ 0.24 & 0.07 $\pm$ 0.07 & 24.90 $\pm$ 5.70 & 12.96 $\pm$ 3.90 \\
 & CPA & 0.80 $\pm$ 0.15 & 0.31 $\pm$ 0.13 & 7.25 $\pm$ 2.70 & 7.65 $\pm$ 5.81 \\
 & scGen & 0.72 $\pm$ 0.20 & 0.15 $\pm$ 0.09 & \textbf{5.28 $\pm$ 4.38} & 7.51 $\pm$ 5.14 \\
 & MintFlow & 0.78 $\pm$ 0.18 & 0.21 $\pm$ 0.16 & 19.99 $\pm$ 1.49 & 8.47 $\pm$ 6.55 \\
 & Cellina (ablated) & 0.79 $\pm$ 0.14 & 0.27 $\pm$ 0.13 & 8.23 $\pm$ 3.53 & 7.92 $\pm$ 5.59 \\
 & Cellina & \textbf{0.81 $\pm$ 0.15} & 0.39 $\pm$ 0.17 & 7.97

2512

***

In [63]:
agg

pearson           direction_match_k  \
                                            mean       std              mean   
perturbation model_name                                                        
Fiber-tracts mean shift                 0.360952  0.238833          0.068000   
             CPA                        0.798208  0.149858          0.309333   
             scGen                      0.715875  0.201174          0.149333   
             MintFlow                   0.779929  0.179957          0.205333   
             Cellina (ablated)          0.785785  0.143455          0.265333   
             Cellina                    0.808807  0.152731          0.390667   
             Cellina-GAT                0.804517  0.157653          0.380000   
             Cellina$_{node-pert}$      0.806143  0.152371          0.396000   
             Cellina-GAT$_{node-pert}$  0.807807  0.156162          0.402857   
             SpatialProp$_{node-pert}$  0.646903  0.180874          0.065333   
Isocortex    mean shift                 0.575376  0.191781          0.250667   
             CPA                        0.837973  0.172484          0.534667   
             scGen                      0.820247  0.142527          0.268000   
             MintFlow                   0.836694  0.164855          0.366667   
             Cellina (ablated)          0.834210  0.159035          0.486667   
             Cellina                    0.853065  0.163093          0.564000   
             Cellina-GAT                0.886631  0.142107          0.635714   
             Cellina$_{node-pert}$      0.846902  0.165859          0.526667   
             Cellina-GAT$_{node-pert}$  0.883319  0.139757          0.601429   
             SpatialProp$_{node-pert}$  0.736170  0.095215          0.076000   

                                                 edistance_pca_log            \
                                             std              mean       std   
perturbation model_name                                                        
Fiber-tracts mean shift                 0.069200         24.896145  5.702891   
             CPA                        0.125554          7.249715  2.701666   
             scGen                      0.090354          5.276147  4.384697   
             MintFlow                   0.162035         19.986863  1.493028   
             Cellina (ablated)          0.126596          8.231004  3.532384   
             Cellina                    0.172356          7.965364  1.441834   
             Cellina-GAT                0.176635          8.224835  1.710762   
             Cellina$_{node-pert}$      0.180507          9.272612  1.999629   
             Cellina-GAT$_{node-pert}$  0.184450          8.080159  1.659169   
             SpatialProp$_{node-pert}$  0.069474         22.352469  3.985128   
Isocortex    mean shift                 0.081369         34.252907  3.978706   
             CPA                        0.136584          6.798150  1.955997   
             scGen                      0.161917          6.209470  5.227868   
             MintFlow                   0.161186         19.172766  1.963619   
             Cellina (ablated)          0.148452          8.832297  5.695179   
             Cellina                    0.138605          7.801514  1.262697   
             Cellina-GAT                0.096454          8.758777  1.415902   
             Cellina$_{node-pert}$      0.140340          8.976245  2.129334   
             Cellina-GAT$_{node-pert}$  0.112991          8.582068  1.144796   
             SpatialProp$_{node-pert}$  0.065987         22.450624  2.325465   

                                         rmse_lfc            
                                             mean       std  
perturbation model_name                                      
Fiber-tracts mean shift                 12.960565  3.896250  
             CPA                         7.645226  5.807097  
             scGen                       7.510745

In [64]:
df_avg = agg.groupby(level="model_name").mean()

In [65]:
df_avg = df_avg.loc[MODEL_ORDER]
df_avg

pearson           direction_match_k            \
                               mean       std              mean       std   
model_name                                                                  
mean shift                 0.468164  0.215307          0.159333  0.075284   
CPA                        0.818090  0.161171          0.422000  0.131069   
scGen                      0.768061  0.171850          0.208667  0.126135   
MintFlow                   0.808311  0.172406          0.286000  0.161610   
Cellina (ablated)          0.809997  0.151245          0.376000  0.137524   
Cellina                    0.830936  0.157912          0.477333  0.155481   
Cellina-GAT                0.845574  0.149880          0.507857  0.136544   
Cellina$_{node-pert}$      0.826523  0.159115          0.461333  0.160423   
Cellina-GAT$_{node-pert}$  0.845563  0.147960          0.502143  0.148721   
SpatialProp$_{node-pert}$  0.691537  0.138045          0.070667  0.067731   

                          edistance_pca_log             rmse_lfc            
                                       mean       std       mean       std  
model_name                                                                  
mean shift                        29.574526  4.840798  12.571237  2.550779  
CPA                                7.023932  2.328831   6.401187  4.755085  
scGen                              5.742809  4.806283   6.525090  4.054144  
MintFlow                          19.579815  1.728324   7.238124  5.409321  
Cellina (ablated)                  8.531650  4.613781   6.803065  4.543992  
Cellina                            7.883439  1.352266   6.231223  4.805637  
Cellina-GAT                        8.491806  1.563332   5.853368  4.423310  
Cellina$_{node-pert}$              9.124428  2.064481   6.330388  4.752672  
Cellina-GAT$_{node-pert}$          8.331113  1.401982   5.880481  4.379427  
SpatialProp$_{node-pert}$         22.401546  3.155296   6.936263  2.557048

In [66]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.2f} $\\pm$ {std:.2f}" if not pd.isna(std) else f"{mean:.2f}"
    return f"\\textbf{{{s}}}" if bold else s

In [67]:
best = find_best(df_avg)
table = pd.DataFrame(index=df_avg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in df_avg.index:
        model_name = model

        mean = df_avg.loc[model, (metric, "mean")]
        std = df_avg.loc[model, (metric, "std")]

        bold = (best.get(metric) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

,Pearson $\uparrow$,Signed Precision $\uparrow$,E-distance $\downarrow$,RMSE\textsubscript{LFC} $\downarrow$
model_name,,,,
mean shift,0.47 $\pm$ 0.22,0.16 $\pm$ 0.08,29.57 $\pm$ 4.84,12.57 $\pm$ 2.55
CPA,0.82 $\pm$ 0.16,0.42 $\pm$ 0.13,7.02 $\pm$ 2.33,6.40 $\pm$ 4.76
scGen,0.77 $\pm$ 0.17,0.21 $\pm$ 0.13,\textbf{5.74 $\pm$ 4.81},6.53 $\pm$ 4.05
MintFlow,0.81 $\pm$ 0.17,0.29 $\pm$ 0.16,19.58 $\pm$ 1.73,7.24 $\pm$ 5.41
Cellina (ablated),0.81 $\pm$ 0.15,0.38 $\pm$ 0.14,8.53 $\pm$ 4.61,6.80 $\pm$ 4.54
Cellina,0.83 $\pm$ 0.16,0.48 $\pm$ 0.16,7.88 $\pm$ 1.35,6.23 $\pm$ 4.81
Cellina-GAT,\textbf{0.85 $\pm$ 0.15},\textbf{0.51 $\pm$ 0.14},8.49 $\pm$ 1.56,\textbf{5.85 $\pm$ 4.42}
Cellina$_{node-pert}$,0.83 $\pm$ 0.16,0.46 $\pm$ 0.16,9.12 $\pm$ 2.06,6.33 $\pm$ 4.75
Cellina-GAT$_{node-pert}$,0.85 $\pm$ 0.15,0.50 $\pm$ 0.15,8.33 $\pm$ 1.40,5.88 $\pm$ 4.38


In [68]:
flat = table.reset_index(level=0, drop=False)

latex = flat.to_latex(
    index=False,
    index_names=True,
    escape=False,
    column_format="l" + "c" * flat.shape[1],
    caption=(
        f"Leave-one-celltype-out performance (top {N_DEG} DEGs). "
        "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
    ),
    label=f"tab:loo_summary_{DATASET_NAME}",
)
latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")

print(latex)

(OUT_DIR / f"loo_summary_{DATASET_NAME}_DEG_{N_DEG}_table_avg.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 50 DEGs). Mean $\pm$ std across slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{lccccc}
\toprule
model_name & Pearson $\uparrow$ & Signed Precision $\uparrow$ & E-distance $\downarrow$ & RMSE\textsubscript{LFC} $\downarrow$ \\
\midrule
mean shift & 0.47 $\pm$ 0.22 & 0.16 $\pm$ 0.08 & 29.57 $\pm$ 4.84 & 12.57 $\pm$ 2.55 \\
CPA & 0.82 $\pm$ 0.16 & 0.42 $\pm$ 0.13 & 7.02 $\pm$ 2.33 & 6.40 $\pm$ 4.76 \\
scGen & 0.77 $\pm$ 0.17 & 0.21 $\pm$ 0.13 & \textbf{5.74 $\pm$ 4.81} & 6.53 $\pm$ 4.05 \\
MintFlow & 0.81 $\pm$ 0.17 & 0.29 $\pm$ 0.16 & 19.58 $\pm$ 1.73 & 7.24 $\pm$ 5.41 \\
Cellina (ablated) & 0.81 $\pm$ 0.15 & 0.38 $\pm$ 0.14 & 8.53 $\pm$ 4.61 & 6.80 $\pm$ 4.54 \\
Cellina & 0.83 $\pm$ 0.16 & 0.48 $\pm$ 0.16 & 7.88 $\pm$ 1.35 & 6.23 $\pm$ 4.81 \\
Cellina-GAT & \textbf{0.85 $\pm$ 0.15} & \textbf{0.51 $\pm$ 0.14} & 8.49 $\pm$ 1.56 & \textbf{5.85 $\pm$ 4.42} \\
Cellina$_{no

1326